# 💳 Card Fraud Triage: Gaussian Mixture Models vs. Deep Autoencoders
**Project 26 — Applied Machine Learning (AML) & Deep Learning (DL)**  
*Team Members:* Ayush Sahu, Rachit Singh, Ashar Asif Shaikh

---

### Project Overview & Problem Statement
In automated credit card fraud detection, fraudulent transactions constitute a minute fraction (approx. 0.17%) of all transactions. Directly training supervised classifiers often struggles with evolving fraud patterns and severe class imbalance.  

This project frames fraud triage under the **unsupervised anomaly detection paradigm**:
1. **Phase 1 (AML Baseline):** Model the probability density of legitimate transactions using a **Gaussian Mixture Model (GMM)**. Unusually low-probability transactions receive high negative log-likelihood (NLL) anomaly scores.
2. **Phase 2 (Deep Learning):** Train an **Undercomplete Denoising Autoencoder (AE)** to reconstruct legitimate transaction manifolds. Unusually high Mean Squared Error (MSE) reconstruction loss flags suspicious transactions.
3. **Operational Evaluation:** Benchmark both approaches using Precision-Recall AUC, controlled False-Positive Rate (FPR) budgets, and fixed Top-K review capacities for human fraud analysts.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Project paths
import sys
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.fraud_detection.comparison import format_comparison_table
from src.fraud_detection.evaluation import format_evaluation_report

print("Libraries loaded successfully! PyTorch version:", torch.__version__)

## 1. Experimental Methodology & Rigor
To ensure zero data leakage and realistic evaluation:
- **Duplicate Removal:** 1,081 exact duplicate transactions were removed (283,726 remaining).
- **Chronological 70/15/15 Split:** Transactions are sorted by  so models are tested on strictly *future* transactions.
- **Legitimate-Only Fitting:** Scalers and models are fitted **exclusively on legitimate training transactions** (). Fraud labels are never exposed during unsupervised training.
- **Validation Threshold Tuning:** Decision thresholds are tuned on the validation set to maximize F1, then applied to the untouched test set.

In [ ]:
# Load saved experimental artifacts
with open(PROJECT_ROOT / "artifacts" / "gmm_results.json") as f:
    gmm_results = json.load(f)

with open(PROJECT_ROOT / "artifacts" / "autoencoder_results.json") as f:
    ae_results = json.load(f)

print("Successfully loaded Phase 1 (GMM) and Phase 2 (Autoencoder) experimental results!")

## 2. Phase 1: Gaussian Mixture Model (AML Baseline)
A GMM models the density as a weighted sum of =38$ Gaussians with diagonal covariance.
The anomaly score is the negative log-likelihood: (x) = -\log p(x)$.

In [ ]:
print(format_evaluation_report(
    "Phase 1: GMM (K=38, Diag Cov)",
    gmm_results["test"],
    fpr_results=gmm_results["fpr_analysis"],
    review_results=gmm_results["review_capacity"],
    threshold=gmm_results["validation"]["threshold"]
))

## 3. Phase 2: Undercomplete Denoising Autoencoder (Deep Learning)
### Architecture:
- **Input:** 30 normalized features (, –, )
- **Encoder:**  	o 20 	o 12 	o 6$ (Bottleneck latent representation)
- **Decoder:**  	o 12 	o 20 	o 30$ (Reconstruct original features)
- **Regularization:** LeakyReLU(0.1), BatchNorm1d, Dropout(0.05), and mild Gaussian input denoising (noise factor 0.02).
- **Anomaly Score:** Mean Squared Error (MSE) reconstruction loss: (x) = rac{1}{D} \sum_{j=1}^D (x_j - \hat{x}_j)^2$.

In [ ]:
print(format_evaluation_report(
    "Phase 2: Undercomplete Autoencoder (Latent Dim = 6)",
    ae_results["test"],
    fpr_results=ae_results["fpr_analysis"],
    review_results=ae_results["review_capacity"],
    threshold=ae_results["validation"]["threshold"]
))

## 4. Head-to-Head Comparative Benchmark
We compare both models on the exact same chronological holdout test set (42,559 transactions; 52 true frauds).

In [ ]:
print(format_comparison_table(gmm_results, ae_results))

## 5. Visualizing the Precision-Recall Trade-off
In extreme class imbalance (0.12% test fraud prevalence), Precision-Recall curves provide the most reliable indicator of detection utility.

In [ ]:
metrics_df = pd.DataFrame({
    "Metric": ["PR-AUC (Average Precision)", "F1-Score", "Precision", "Recall"],
    "Phase 1 (GMM)": [
        gmm_results["test"]["average_precision"],
        gmm_results["test"]["f1"],
        gmm_results["test"]["precision"],
        gmm_results["test"]["recall"]
    ],
    "Phase 2 (Autoencoder)": [
        ae_results["test"]["average_precision"],
        ae_results["test"]["f1"],
        ae_results["test"]["precision"],
        ae_results["test"]["recall"]
    ]
})

fig, ax = plt.subplots(figsize=(10, 5), dpi=130)
x = np.arange(len(metrics_df))
width = 0.35

rects1 = ax.bar(x - width/2, metrics_df["Phase 1 (GMM)"], width, label="Phase 1: GMM K=38", color="#3182bd")
rects2 = ax.bar(x + width/2, metrics_df["Phase 2 (Autoencoder)"], width, label="Phase 2: Autoencoder", color="#e6550d")

ax.set_ylabel("Score", fontsize=12)
ax.set_title("Test Set Comparison: GMM vs. Autoencoder", fontsize=14, fontweight="bold", pad=12)
ax.set_xticks(x)
ax.set_xticklabels(metrics_df["Metric"], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.0)
ax.grid(axis="y", linestyle=":", alpha=0.7)

for rects in [rects1, rects2]:
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f"{h:.2%}",
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=10, fontweight="semibold")

plt.tight_layout()
plt.show()

## 6. Key Conclusions & Student Discussion

1. **Why Autoencoder Outperforms GMM:**
   - **The Gaussian Assumption Gap:** A GMM assumes that the legitimate transaction space can be approximated by a combination of elliptical Gaussian clusters. In 30 dimensions with complex non-linear transaction interactions (e.g.  vs  vs PCA combinations), diagonal Gaussians struggle to capture curved manifolds.
   - **Non-Linear Dimensionality Reduction:** The Autoencoder uses non-linear activations () and bottleneck compression ( 	o 20 	o 12 	o 6$) to learn the intrinsic low-dimensional manifold of legitimate spending. Fraudulent transactions that diverge from this manifold suffer severe reconstruction degradation.

2. **Operational Triage Benefits:**
   - At a strict **Top-50 daily review capacity**, the Autoencoder catches **36 out of 52 frauds (72.0% precision)**, compared to **24 frauds (48.0% precision)** for the GMM.
   - At a **0.1% false-positive rate budget**, the Autoencoder catches **75.0%** of frauds with only 32 false alarms, while GMM achieves **40.4%** recall.

3. **Final Verdict:**
   Both unsupervised methods avoid overfitting to historical fraud instances, but the Deep Learning Autoencoder provides significantly higher operational triage efficiency for fraud investigation teams.